# MLB API Inspection & Data Structure Analysis

Inspiration: WBC backfill (`../WBC/src/ingestion/`). This notebook gathers correct API IDs, inspects data structures, and recommends efficient warehouse storage for the MLB backfill.

**Goals:**
1. Document correct `sportId`, `leagueId`, `gameType`, and stage mappings
2. Inspect raw API structures (schedule, boxscore, linescore, playByPlay, **feed/live**)
3. Document **sabermetric data** (velocity, spin, launch angle) in feed/live — like LIDOM
4. Recommend optimal data types for warehouse (large data volume)

## 1. API Constants – sportId, gameType, Stage Mapping

MLB Stats API base: `https://statsapi.mlb.com/api/v1`

In [83]:
import json
import requests
from pathlib import Path

BASE_URL = "https://statsapi.mlb.com/api/v1"

# === MLB vs WBC (reference) ===
# WBC: sportId=51, leagueId=160 (WBC_LEAGUE_ID)
# MLB: sportId=1, leagues 103 (AL) and 104 (NL)

SPORT_ID_MLB = 1
SPORT_ID_WBC = 51  # reference from WBC backfill

# Fetch game types and sports from API (canonical source)
game_types = requests.get(f"{BASE_URL}/gameTypes", timeout=10).json()
sports = requests.get(f"{BASE_URL}/sports", timeout=10).json()

print("=== gameTypes (from /api/v1/gameTypes) ===")
for gt in game_types:
    print(f"  {gt['id']:>2} = {gt['description']}")

print("\n=== MLB-relevant sports ===")
for s in sports.get("sports", []):
    if s["id"] in (1, 51):  # MLB and WBC
        print(f"  sportId={s['id']}: {s['name']} ({s['abbreviation']})")

=== gameTypes (from /api/v1/gameTypes) ===
   S = Spring Training
   R = Regular Season
   F = Wild Card
   D = Division Series
   L = League Championship Series
   W = World Series
   C = Championship
   N = Nineteenth Century Series
   P = Playoffs
   A = All-Star Game
   I = Intrasquad
   E = Exhibition

=== MLB-relevant sports ===
  sportId=1: Major League Baseball (MLB)
  sportId=51: International Baseball (INT)


In [84]:
# === gameType → warehouse stage mapping ===
# Our setup_warehouse uses: spring_training, regular_season, all_star, playoffs
# Playoffs has subfolders: wild_card, division, championship, world_series

GAME_TYPE_TO_STAGE = {
    "S": "spring_training",   # Spring Training
    "R": "regular_season",    # Regular Season
    "A": "all_star",          # All-Star Game
    "P": "playoffs",          # Playoffs (generic)
    "F": "playoffs/wild_card",      # Wild Card
    "D": "playoffs/division",       # Division Series
    "L": "playoffs/championship",   # League Championship Series
    "W": "playoffs/world_series",   # World Series
    "C": "playoffs/championship",   # Championship (alternate)
}
# E=Exhibition, I=Intrasquad, N=Nineteenth Century – typically skip for MLB warehouse

print("gameType → warehouse stage:")
for code, stage in GAME_TYPE_TO_STAGE.items():
    desc = next((g["description"] for g in game_types if g["id"] == code), "?")
    print(f"  {code} ({desc}) → {stage}")

gameType → warehouse stage:
  S (Spring Training) → spring_training
  R (Regular Season) → regular_season
  A (All-Star Game) → all_star
  P (Playoffs) → playoffs
  F (Wild Card) → playoffs/wild_card
  D (Division Series) → playoffs/division
  L (League Championship Series) → playoffs/championship
  W (World Series) → playoffs/world_series
  C (Championship) → playoffs/championship


## 2. Schedule API – Parameters & Sample Fetch

**Schedule endpoint:** `GET /schedule`

**Key parameters:**
- `sportId=1` — MLB (required)
- `season=2024` — full season, returns all game types
- `gameType=S|R|A|P|F|D|L|W` — filter by type
- `startDate` / `endDate` — alternative to season (YYYY-MM-DD)
- `hydrate=team` — expand team info (recommended)

In [85]:
def get_schedule(sport_id: int, season: int = None, game_type: str = None, 
                 start_date: str = None, end_date: str = None) -> dict | None:
    url = f"{BASE_URL}/schedule"
    params = {"sportId": sport_id, "hydrate": "team"}
    if season:
        params["season"] = season
    if game_type:
        params["gameType"] = game_type
    if start_date:
        params["startDate"] = start_date
    if end_date:
        params["endDate"] = end_date
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    return r.json()

# Sample: 2024 regular season (gameType=R)
sched_r = get_schedule(SPORT_ID_MLB, season=2024, game_type="R")
print(f"2024 Regular Season: {sched_r['totalGames']} games")
# Sample: 2024 spring training
sched_s = get_schedule(SPORT_ID_MLB, season=2024, game_type="S")
print(f"2024 Spring Training: {sched_s['totalGames']} games")

2024 Regular Season: 2469 games
2024 Spring Training: 472 games


In [86]:
# Schedule response structure
dates = sched_r.get("dates", [])
first_date = dates[0] if dates else {}
first_game = first_date.get("games", [{}])[0] if first_date.get("games") else {}

print("=== Schedule top-level keys ===")
print(list(sched_r.keys()))
print("\n=== First game keys ===")
print(list(first_game.keys()))
print("\n=== First game sample (selected fields) ===")
print({
    "gamePk": first_game.get("gamePk"),
    "gameType": first_game.get("gameType"),
    "season": first_game.get("season"),
    "officialDate": first_game.get("officialDate"),
    "status": first_game.get("status", {}).get("abstractGameState"),
    "venue": first_game.get("venue", {}).get("name"),
    "teams.away.team.id": first_game.get("teams", {}).get("away", {}).get("team", {}).get("id"),
    "teams.away.team.name": first_game.get("teams", {}).get("away", {}).get("team", {}).get("name"),
    "teams.home.team.id": first_game.get("teams", {}).get("home", {}).get("team", {}).get("id"),
    "teams.home.team.name": first_game.get("teams", {}).get("home", {}).get("team", {}).get("name"),
    "teams.away.score": first_game.get("teams", {}).get("away", {}).get("score"),
    "teams.home.score": first_game.get("teams", {}).get("home", {}).get("score"),
})

=== Schedule top-level keys ===
['copyright', 'totalItems', 'totalEvents', 'totalGames', 'totalGamesInProgress', 'dates']

=== First game keys ===
['gamePk', 'gameGuid', 'link', 'gameType', 'season', 'gameDate', 'officialDate', 'status', 'teams', 'venue', 'content', 'isTie', 'gameNumber', 'publicFacing', 'doubleHeader', 'gamedayType', 'tiebreaker', 'calendarEventID', 'seasonDisplay', 'dayNight', 'description', 'scheduledInnings', 'reverseHomeAwayStatus', 'inningBreakLength', 'gamesInSeries', 'seriesGameNumber', 'seriesDescription', 'recordSource', 'ifNecessary', 'ifNecessaryDescription']

=== First game sample (selected fields) ===
{'gamePk': 745444, 'gameType': 'R', 'season': '2024', 'officialDate': '2024-03-20', 'status': 'Final', 'venue': 'Gocheok Sky Dome', 'teams.away.team.id': 119, 'teams.away.team.name': 'Los Angeles Dodgers', 'teams.home.team.id': 135, 'teams.home.team.name': 'San Diego Padres', 'teams.away.score': 5, 'teams.home.score': 2}


## 3. Game Endpoints – Boxscore, Linescore, PlayByPlay, **Feed/Live (RAW)**

Per-game endpoints:
- `GET /game/{gamePk}/boxscore` — batting/pitching stats per player
- `GET /game/{gamePk}/linescore` — inning-by-inning runs/hits/errors
- `GET /game/{gamePk}/playByPlay` — every play and pitch (no velocity/spin)
- **`GET /api/v1.1/game/{gamePk}/feed/live`** — **full raw feed with Statcast/sabermetric data** (velocity, spin, launch angle, etc.) — **SAVE THIS AS RAW JSON**

In [87]:
# Pick a regular-season game (not spring training) for realistic structure
def first_game_pk(sched):
    for d in sched.get("dates", []):
        for g in d.get("games", []):
            return g.get("gamePk")
    return None

game_pk = first_game_pk(sched_r) or 745444  # fallback: 2024 LAD @ SD

box = requests.get(f"{BASE_URL}/game/{game_pk}/boxscore", timeout=15).json()
line = requests.get(f"{BASE_URL}/game/{game_pk}/linescore", timeout=15).json()
pbp = requests.get(f"{BASE_URL}/game/{game_pk}/playByPlay", timeout=15).json()

def json_size(obj):
    return len(json.dumps(obj).encode("utf-8")) / 1024

print(f"gamePk={game_pk}")
print(f"  boxscore size:    {json_size(box):.1f} KB")
print(f"  linescore size:   {json_size(line):.1f} KB")
print(f"  playByPlay size:  {json_size(pbp):.1f} KB")

gamePk=745444
  boxscore size:    204.2 KB
  linescore size:   3.5 KB
  playByPlay size:  589.9 KB


In [88]:
# Boxscore structure (same as WBC)
print("=== Boxscore top keys ===")
print(list(box.keys()))
for side in ("away", "home"):
    tn = box.get("teams", {}).get(side, {})
    print(f"\n--- teams.{side} keys: {list(tn.keys())}")
    team = tn.get("team", {})
    players = tn.get("players", {})
    pitchers = tn.get("pitchers", [])
    batters = tn.get("batters", [])
    print(f"  team: id={team.get('id')}, name={team.get('name')}")
    print(f"  pitchers: {len(pitchers)} IDs, batters: {len(batters)} IDs")
    print(f"  players dict: {len(players)} entries (keyed by ID{{personId}})")
    if players:
        sample_key = list(players.keys())[0]
        p = players[sample_key]
        print(f"  sample player keys: {list(p.keys())}")
        if "stats" in p and "pitching" in p["stats"]:
            print(f"  pitching stat keys: {list(p['stats']['pitching'].keys())[:10]}...")

=== Boxscore top keys ===
['copyright', 'teams', 'officials', 'info', 'pitchingNotes', 'topPerformers']

--- teams.away keys: ['team', 'teamStats', 'players', 'batters', 'pitchers', 'bench', 'bullpen', 'battingOrder', 'info', 'note']
  team: id=119, name=Los Angeles Dodgers
  pitchers: 5 IDs, batters: 15 IDs
  players dict: 29 entries (keyed by ID{personId})
  sample player keys: ['person', 'jerseyNumber', 'position', 'status', 'parentTeamId', 'battingOrder', 'stats', 'seasonStats', 'gameStatus', 'allPositions']
  pitching stat keys: []...

--- teams.home keys: ['team', 'teamStats', 'players', 'batters', 'pitchers', 'bench', 'bullpen', 'battingOrder', 'info', 'note']
  team: id=135, name=San Diego Padres
  pitchers: 8 IDs, batters: 19 IDs
  players dict: 29 entries (keyed by ID{personId})
  sample player keys: ['person', 'jerseyNumber', 'position', 'status', 'parentTeamId', 'battingOrder', 'stats', 'seasonStats', 'gameStatus', 'allPositions']
  pitching stat keys: []...


In [89]:
# Linescore structure (compact)
print("=== Linescore keys ===")
print(list(line.keys()))
print("\n innings[0] (per-inning):", line.get("innings", [{}])[0])
print("\n teams.home/away:", {k: line.get("teams", {}).get(k) for k in ("home", "away")})

=== Linescore keys ===
['copyright', 'currentInning', 'currentInningOrdinal', 'inningState', 'inningHalf', 'isTopInning', 'scheduledInnings', 'innings', 'teams', 'defense', 'offense', 'balls', 'strikes', 'outs']

 innings[0] (per-inning): {'num': 1, 'ordinalNum': '1st', 'home': {'runs': 0, 'hits': 0, 'errors': 0, 'leftOnBase': 0}, 'away': {'runs': 0, 'hits': 0, 'errors': 0, 'leftOnBase': 1}}

 teams.home/away: {'home': {'runs': 2, 'hits': 4, 'errors': 2, 'leftOnBase': 5, 'isWinner': False}, 'away': {'runs': 5, 'hits': 7, 'errors': 0, 'leftOnBase': 13, 'isWinner': True}}


In [90]:
# PlayByPlay structure (largest – ~100+ plays per game, each with playEvents = pitches)
all_plays = pbp.get("allPlays", [])
print(f"=== PlayByPlay: {len(all_plays)} allPlays (at-bats) ===")
print("First play keys:", list(all_plays[0].keys()) if all_plays else [])
if all_plays:
    p0 = all_plays[0]
    print("\nFirst play sample:")
    print("  result:", p0.get("result", {}))
    print("  matchup.batter:", p0.get("matchup", {}).get("batter"))
    print("  matchup.pitcher:", p0.get("matchup", {}).get("pitcher"))
    events = p0.get("playEvents", [])
    pitches = [e for e in events if e.get("isPitch")]
    print(f"  playEvents: {len(events)} total, {len(pitches)} pitches")

=== PlayByPlay: 79 allPlays (at-bats) ===
First play keys: ['result', 'about', 'count', 'matchup', 'pitchIndex', 'actionIndex', 'runnerIndex', 'runners', 'playEvents', 'playEndTime', 'atBatIndex']

First play sample:
  result: {'type': 'atBat', 'event': 'Walk', 'eventType': 'walk', 'description': 'Mookie Betts walks.', 'rbi': 0, 'awayScore': 0, 'homeScore': 0, 'isOut': False}
  matchup.batter: {'id': 605141, 'fullName': 'Mookie Betts', 'link': '/api/v1/people/605141'}
  matchup.pitcher: {'id': 506433, 'fullName': 'Yu Darvish', 'link': '/api/v1/people/506433'}
  playEvents: 8 total, 4 pitches


## 3b. Feed/Live – Raw JSON with Sabermetric Data (CRITICAL)

**LIDOM uses this endpoint** (`data/warehouse/lidom_api/*/raw/*_pbp.json`). Same structure for MLB.

The **feed/live** endpoint returns `liveData.plays.allPlays` with `playEvents` where each pitch has:
- **pitchData**: `startSpeed`, `endSpeed`, `breaks` (spinRate, breakVertical, breakHorizontal, etc.), `zone`, `plateTime`, `extension`
- **hitData** (batted balls): `launchSpeed`, `launchAngle`, `totalDistance`, `trajectory`, `hardness`, `location`, `coordinates`

**Always save feed/live as raw JSON** — it contains ALL sabermetric variables. playByPlay alone does NOT include velocity/spin.

In [91]:
# Fetch feed/live (the raw that matters for sabermetrics)
FEED_URL = "https://statsapi.mlb.com/api/v1.1/game/{game_pk}/feed/live"
feed = requests.get(FEED_URL.format(game_pk=game_pk), timeout=30).json()

print(f"feed/live size: {json_size(feed):.1f} KB")
print("Top-level keys:", list(feed.keys()))
plays = feed.get("liveData", {}).get("plays", {}).get("allPlays", [])
print(f"Plays (at-bats): {len(plays)}")

# Extract pitchData and hitData structure from first few pitches
pitch_data_keys = set()
hit_data_keys = set()
sample_pitch = None
sample_hit = None
for p in plays[:10]:
    for e in p.get("playEvents", []):
        if e.get("isPitch") and "pitchData" in e:
            pitch_data_keys.update(e["pitchData"].keys())
            if sample_pitch is None:
                sample_pitch = e["pitchData"]
        if "hitData" in e:
            hit_data_keys.update(e["hitData"].keys())
            if sample_hit is None:
                sample_hit = e["hitData"]

print("\n=== pitchData (PITCHING SABERMETRICS) ===")
print("Keys:", sorted(pitch_data_keys))
if sample_pitch:
    print("Sample:", {k: sample_pitch.get(k) for k in ["startSpeed", "endSpeed", "zone", "plateTime"]})
    print("breaks:", sample_pitch.get("breaks"))

print("\n=== hitData (BATTING SABERMETRICS) ===")
print("Keys:", sorted(hit_data_keys))
if sample_hit:
    print("Sample:", sample_hit)

feed/live size: 859.3 KB
Top-level keys: ['copyright', 'gamePk', 'link', 'metaData', 'gameData', 'liveData']
Plays (at-bats): 79

=== pitchData (PITCHING SABERMETRICS) ===
Keys: ['breaks', 'coordinates', 'endSpeed', 'extension', 'plateTime', 'startSpeed', 'strikeZoneBottom', 'strikeZoneTop', 'typeConfidence', 'zone']
Sample: {'startSpeed': 94.5, 'endSpeed': 85.8, 'zone': 12, 'plateTime': 0.39965900778770447}
breaks: {'breakAngle': 22.8, 'breakLength': 3.6, 'breakY': 24.0, 'breakVertical': -13.4, 'breakVerticalInduced': 17.4, 'breakHorizontal': 6.2, 'spinRate': 2394, 'spinDirection': 200}

=== hitData (BATTING SABERMETRICS) ===
Keys: ['coordinates', 'hardness', 'launchAngle', 'launchSpeed', 'location', 'totalDistance', 'trajectory']
Sample: {'launchSpeed': 77.9, 'launchAngle': -13.0, 'totalDistance': 11.0, 'trajectory': 'ground_ball', 'hardness': 'medium', 'location': '6', 'coordinates': {'coordX': 116.82, 'coordY': 150.37}}


In [92]:
# Raw storage recommendation (like LIDOM: data/warehouse/lidom_api/{season}/raw/)
# MLB: data/warehouse/mlb/raw/{season}/{stage}/game_{game_pk}_{date}_feed_live.json

SABERMETRIC_PITCH_FIELDS = [
    "startSpeed", "endSpeed", "zone", "plateTime", "extension",
    "breaks",  # spinRate, breakVertical, breakHorizontal, breakAngle, breakLength, etc.
]
SABERMETRIC_HIT_FIELDS = [
    "launchSpeed", "launchAngle", "totalDistance",
    "trajectory", "hardness", "location", "coordinates",
]
print("Pitch sabermetrics:", SABERMETRIC_PITCH_FIELDS)
print("Hit sabermetrics:", SABERMETRIC_HIT_FIELDS)

Pitch sabermetrics: ['startSpeed', 'endSpeed', 'zone', 'plateTime', 'extension', 'breaks']
Hit sabermetrics: ['launchSpeed', 'launchAngle', 'totalDistance', 'trajectory', 'hardness', 'location', 'coordinates']


In [93]:
# Save sample raw feed to outputs (for inspection)
out_dir = Path("..") / "outputs"
out_dir.mkdir(exist_ok=True)
sample_raw_path = out_dir / f"sample_feed_live_{game_pk}.json"
with open(sample_raw_path, "w") as f:
    json.dump(feed, f, indent=2)
print("Sample raw feed saved:", sample_raw_path.resolve())

Sample raw feed saved: /Users/gilrojasb/Desktop/Mallitalytics_VS/MLB/outputs/sample_feed_live_745444.json


## 4. Volume Estimates & Efficient Data Types

**Scale:** ~2,430 regular-season games/year + ~220 spring + ~40 playoffs + 1 all-star ≈ **2,700 games/year**

| Endpoint   | Est. size/game | Est. total/year | Notes                          |
|------------|----------------|-----------------|--------------------------------|
| Schedule   | ~500 KB (full season in one call) | 1 call/season  | Small; store parsed CSV/Parquet |
| Boxscore   | 40–80 KB       | ~150 MB         | Normalize to tables            |
| Linescore  | 2–5 KB         | ~10 MB          | Compact                        |
| **feed/live** | **150–400 KB** | **~800 MB**  | **RAW – velocity, spin, launch; SAVE** |
| PlayByPlay | 80–200 KB      | ~300 MB         | Subset (no Statcast)           |

**Recommended storage:**
- **Parquet** over CSV for boxscore-derived tables (game_pitching, game_batting) — columnar, compressible
- **Optimal dtypes:** `int32` for IDs, `int16` or `int8` for counts (hits, runs, strikeouts), `float32` for rates
- **Raw JSON (feed/live):** Save full JSON for sabermetrics; gzip for storage efficiency

In [94]:
# Recommended dtypes for warehouse (pandas/Parquet)
# Mirrors WBC schema but with MLB-specific stages

RECOMMENDED_DTYPES = {
    "schedule_games": {
        "game_pk": "int64",      # primary key
        "season": "int16",
        "game_type": "category", # S,R,A,P,F,D,L,W
        "game_date": "str",      # or datetime64
        "venue_id": "int32",
        "away_team_id": "int32",
        "home_team_id": "int32",
        "away_score": "int8",    # 0–30 typical
        "home_score": "int8",
        "is_final": "bool",
    },
    "game_pitching": {
        "game_pk": "int64",
        "season": "int16",
        "team_side": "category",
        "team_id": "int32",
        "person_id": "int64",
        "innings_pitched": "str",  # "6.1" etc – keep as string or float32
        "hits": "int8", "runs": "int8", "earned_runs": "int8",
        "base_on_balls": "int8", "strike_outs": "int8", "home_runs": "int8",
        "number_of_pitches": "int16", "strikes": "int16",
        "wins": "int8", "losses": "int8", "saves": "int8",
    },
    "game_batting": {
        "game_pk": "int64", "season": "int16", "team_side": "category",
        "team_id": "int32", "person_id": "int64", "batting_order": "int8",
        "at_bats": "int8", "runs": "int8", "hits": "int8",
        "doubles": "int8", "triples": "int8", "home_runs": "int8",
        "rbi": "int8", "base_on_balls": "int8", "strike_outs": "int8",
        "stolen_bases": "int8",
    },
    "linescore_innings": {
        "game_pk": "int64", "inning": "int8",
        "away_runs": "int8", "away_hits": "int8", "away_errors": "int8",
        "home_runs": "int8", "home_hits": "int8", "home_errors": "int8",
    },
}
print("Recommended dtypes defined. Use pd.DataFrame.astype() when building tables.")

Recommended dtypes defined. Use pd.DataFrame.astype() when building tables.


## 5. Summary – API Reference for Backfill

| Item | Value |
|------|-------|
| **Base URL** | `https://statsapi.mlb.com/api/v1` |
| **sportId (MLB)** | `1` |
| **Schedule params** | `sportId=1`, `season=YYYY`, `gameType=S\|R\|A\|P\|F\|D\|L\|W` |
| **gameType → stage** | S→spring_training, R→regular_season, A→all_star, F/D/L/W→playoffs/* |
| **Endpoints** | `/schedule`, `/game/{gamePk}/boxscore`, `/game/{gamePk}/linescore`, `/game/{gamePk}/playByPlay` |
| **RAW (sabermetrics)** | `/api/v1.1/game/{gamePk}/feed/live` — velocity, spin, launch angle |

**Next steps:** Implement `load_mlb_warehouse.py` (patterned after WBC) — save **feed/live** as raw JSON (sabermetrics), extract to Parquet with dtypes above.

In [95]:
# Optional: save inspection report for reference
out_dir = Path("..") / "outputs"  # MLB/outputs
out_dir.mkdir(exist_ok=True)
report = {
    "sportId": SPORT_ID_MLB,
    "gameTypeToStage": GAME_TYPE_TO_STAGE,
    "scheduleSample": {
        "totalGames_regular_2024": sched_r.get("totalGames"),
        "totalGames_spring_2024": sched_s.get("totalGames"),
    },
    "sampleGamePk": game_pk,
    "payloadSizesKB": {
        "boxscore": round(json_size(box), 1),
        "linescore": round(json_size(line), 1),
        "playByPlay": round(json_size(pbp), 1),
        "feed_live": round(json_size(feed), 1),
    },
    "sabermetricPitchFields": SABERMETRIC_PITCH_FIELDS,
    "sabermetricHitFields": SABERMETRIC_HIT_FIELDS,
    "recommendedDtypes": {k: list(v.keys()) for k, v in RECOMMENDED_DTYPES.items()},
}
report_path = out_dir / "mlb_inspection_report.json"
with open(report_path, "w") as f:
    json.dump(report, f, indent=2)
print("Saved:", report_path.resolve())

Saved: /Users/gilrojasb/Desktop/Mallitalytics_VS/MLB/outputs/mlb_inspection_report.json


---
# 6. Statcast / Baseball Savant – Datos que NO vienen en el feed

**Contexto:** LIDOM no tenía Statcast. MLB sí. Algunos datos avanzados están en Baseball Savant pero no en el Stats API feed.

| Tenemos (feed/live) | Nos falta (Baseball Savant) |
|---------------------|-----------------------------|
| launchSpeed, launchAngle, spin, velocity | **xwOBA** (estimated_woba_using_speedangle) |
| pitchData, hitData completos | **wOBA** (woba_value, woba_denom) |
| — | **Bat speed** (swing speed, desde 2023–24) |
| — | estimated_ba_using_speedangle, launch_speed_angle (barrels) |

**Fuentes:**
- [Baseball Savant Statcast Search](https://baseballsavant.mlb.com/statcast_search) – CSV manual
- [Bat Tracking Leaderboard](https://baseballsavant.mlb.com/leaderboard/bat-tracking) – bat speed (CSV)
- **pybaseball** – librería Python para bajar Statcast por fecha o por juego

In [71]:
# Instalar pybaseball si no lo tienes: pip install pybaseball
# Empezamos suave: un solo juego para ver qué columnas trae

try:
    from pybaseball import statcast_single_game
except ImportError:
    print("Instala: pip install pybaseball")
    raise

# Probar con un game_pk que ya tenemos en raw (745444 o 776498)
game_pk_test = 776498  # LAD @ SD 2024
df = statcast_single_game(game_pk_test)

if df is not None and len(df) > 0:
    print(f"Statcast rows: {len(df)}")
    print("\nColumnas relevantes (wOBA, xwOBA, launch):")
    woba_cols = [c for c in df.columns if 'woba' in c.lower() or 'estimated' in c.lower() or 'launch' in c.lower()]
    print(woba_cols)
    if woba_cols:
        print("\nMuestra (primeras 3 filas):")
        print(df[woba_cols].head(3).to_string())
else:
    print("No data returned (puede ser juego antiguo o formato cambiado)")

Statcast rows: 255

Columnas relevantes (wOBA, xwOBA, launch):
['launch_speed', 'launch_angle', 'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'launch_speed_angle', 'estimated_slg_using_speedangle']

Muestra (primeras 3 filas):
    launch_speed  launch_angle  estimated_ba_using_speedangle  estimated_woba_using_speedangle  woba_value  woba_denom  launch_speed_angle  estimated_slg_using_speedangle
80          86.2          26.0                          0.109                            0.125         0.0         1.0                 3.0                           0.188
82           NaN           NaN                            NaN                            0.000         0.0         1.0                 NaN                             NaN
89          84.5          48.0                            NaN                              NaN         NaN         NaN                 NaN                             NaN


### Integración futura (idea)

1. **feed/live** → raw JSON por juego (ya lo tenemos)
2. **pybaseball statcast_single_game(game_pk)** → DataFrame con xwOBA, woba_value, etc.
3. **Join:** Statcast tiene `sv_id` (play event ID) y `game_pk` — se puede cruzar con playEvents por inning, atBatIndex, pitchNumber para enriquecer cada pitch/batted ball.

Bat speed: por ahora solo en Baseball Savant Bat Tracking (CSV manual o scraping). Explorar después.

---
# 7. Exploración: Integración Statcast + feed/live (empezando suave)

**Contexto:** Nunca hemos trabajado con Statcast al nivel que vamos aquí. En LIDOM no había. Vamos paso a paso.

**Plan suave:**
1. Extraer la estructura pitch-level de nuestro feed/live raw (ya lo tenemos guardado)
2. Identificar las **claves de join** (game_pk, inning, atBatIndex, pitchNumber) para cruzar con Statcast
3. Probar pybaseball cuando esté instalado
4. Documentar qué columnas agregaría Statcast (xwOBA, woba_value, etc.)

In [72]:
# Paso 1: Extraer pitches del feed/live que ya tenemos
# Sin dependencias externas - solo nuestro raw JSON

import json
from pathlib import Path

feed_path = Path("../outputs/sample_feed_live_776498.json")  # MIA @ WSH 2025
with open(feed_path) as f:
    feed = json.load(f)

game_pk = feed["gamePk"]
all_plays = feed["liveData"]["plays"]["allPlays"]

pitches = []
for play in all_plays:
    ab_idx = play["atBatIndex"]
    inn = play["about"]["inning"]
    half = play["about"]["halfInning"]  # "top" | "bottom"
    for ev in play.get("playEvents", []):
        if not ev.get("isPitch") or "pitchData" not in ev:
            continue
        pd = ev["pitchData"]
        row = {
            "game_pk": game_pk,
            "inning": inn,
            "half_inning": half,
            "at_bat_index": ab_idx,
            "pitch_number": ev.get("pitchNumber"),
            "start_speed": pd.get("startSpeed"),
            "zone": pd.get("zone"),
            "play_id": ev.get("playId"),
        }
        if "breaks" in pd:
            row["spin_rate"] = pd["breaks"].get("spinRate")
        if "hitData" in ev:
            hd = ev["hitData"]
            row["launch_speed"] = hd.get("launchSpeed")
            row["launch_angle"] = hd.get("launchAngle")
        pitches.append(row)

print(f"Game {game_pk}: {len(pitches)} pitches extraídos")
print("\nClaves de join para Statcast: game_pk, inning, at_bat_index, pitch_number")
print("\nMuestra (primeros 3):")
for p in pitches[:3]:
    print(p)

Game 776498: 255 pitches extraídos

Claves de join para Statcast: game_pk, inning, at_bat_index, pitch_number

Muestra (primeros 3):
{'game_pk': 776498, 'inning': 1, 'half_inning': 'top', 'at_bat_index': 0, 'pitch_number': 1, 'start_speed': 92.5, 'zone': 3, 'play_id': '4b10184d-1d8f-33bd-bdc7-c9ef19a17458', 'spin_rate': 2316}
{'game_pk': 776498, 'inning': 1, 'half_inning': 'top', 'at_bat_index': 0, 'pitch_number': 2, 'start_speed': 90.5, 'zone': 11, 'play_id': '6ea57801-1e93-3194-b425-e7c4b8276760', 'spin_rate': 2293}
{'game_pk': 776498, 'inning': 1, 'half_inning': 'top', 'at_bat_index': 0, 'pitch_number': 3, 'start_speed': 82.9, 'zone': 2, 'play_id': '86eece98-e27c-3d77-b441-e2581baaf25c', 'spin_rate': 2603}


In [73]:
# Paso 2: ¿Qué columnas trae Statcast? (requiere: pip install pybaseball)
# Statcast ordena por: game_date, game_pk, at_bat_number, pitch_number

try:
    from pybaseball import statcast_single_game
    
    df_sc = statcast_single_game(game_pk)
    if df_sc is not None and len(df_sc) > 0:
        print(f"Statcast: {len(df_sc)} filas (pitch-level)")
        woba_cols = [c for c in df_sc.columns if "woba" in c.lower() or "estimated" in c.lower()]
        print("\nColumnas wOBA/xwOBA:", woba_cols)
        # Claves de join en Statcast
        key_cols = [c for c in df_sc.columns if any(x in c.lower() for x in ["game", "inning", "at_bat", "pitch"])]
        print("\nClaves potenciales:", key_cols[:15])
    else:
        print("Statcast devolvió vacío (juego puede ser muy reciente o sin datos)")
except ImportError:
    print("pybaseball no instalado. Ejecuta: pip install pybaseball")

Statcast: 255 filas (pitch-level)

Columnas wOBA/xwOBA: ['estimated_ba_using_speedangle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'estimated_slg_using_speedangle']

Claves potenciales: ['pitch_type', 'game_date', 'pitcher', 'game_type', 'game_year', 'inning', 'inning_topbot', 'game_pk', 'at_bat_number', 'pitch_number', 'pitch_name', 'delta_pitcher_run_exp', 'n_thruorder_pitcher', 'n_priorpa_thisgame_player_at_bat', 'pitcher_days_since_prev_game']


---
# 8. Integración MLB API + Statcast — Enigma resuelto

**Objetivo:** Combinar feed/live (MLB Stats API) con Statcast (pybaseball) de forma que sea fácil extraer para contenido, como en LIDOM.

**Arquitectura propuesta (como LIDOM):**
- **Raw:** feed/live JSON por juego (diamante sin pulir)
- **Procesado:** pitches_enriched = feed pitches + Statcast (join por game_pk, inning, at_bat_index, pitch_number)
- **Content:** Una función `load_game_pitches(game_pk)` que devuelve el DataFrame listo para crear contenido

In [74]:
# Paso 1: Join real — feed pitches + Statcast
# Requiere que hayas ejecutado las celdas anteriores (pitches, df_sc, game_pk)

import pandas as pd

df_feed = pd.DataFrame(pitches)
# Statcast usa at_bat_number; nosotros tenemos at_bat_index (mismo concepto)
df_sc_renamed = df_sc.rename(columns={"at_bat_number": "at_bat_index"})

# Columnas Statcast útiles para contenido (evitar 100+ columnas)
STATCAST_COLS_FOR_CONTENT = [
    "game_pk", "inning", "at_bat_index", "pitch_number",
    "estimated_woba_using_speedangle", "woba_value", "woba_denom",
    "estimated_ba_using_speedangle", "launch_speed_angle",
    "pitch_name", "pitcher", "batter",
    "release_speed", "release_spin_rate",
    "zone", "type", "description", "events", "bb_type",
]
statcast_cols = [c for c in STATCAST_COLS_FOR_CONTENT if c in df_sc_renamed.columns]
df_sc_subset = df_sc_renamed[statcast_cols].copy()

# Merge: feed es base, Statcast agrega columnas (suffix _sc si hay solapamiento)
merged = df_feed.merge(
    df_sc_subset,
    on=["game_pk", "inning", "at_bat_index", "pitch_number"],
    how="left",
    suffixes=("", "_sc"),
)
# Limpiar duplicados de columna (ej. zone_sc si existió)
merged = merged.loc[:, ~merged.columns.duplicated()]

print(f"Join: {len(df_feed)} pitches → {len(merged)} (left join)")
print("\nColumnas en merged:", list(merged.columns))
print("\nMuestra (velocidad, xwOBA, barrels):")
show_cols = ["inning", "at_bat_index", "pitch_number", "start_speed", "launch_speed", "estimated_woba_using_speedangle", "launch_speed_angle"]
show_cols = [c for c in show_cols if c in merged.columns]
print(merged[show_cols].head(8).to_string())

Join: 255 pitches → 255 (left join)

Columnas en merged: ['game_pk', 'inning', 'half_inning', 'at_bat_index', 'pitch_number', 'start_speed', 'zone', 'play_id', 'spin_rate', 'launch_speed', 'launch_angle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'estimated_ba_using_speedangle', 'launch_speed_angle', 'pitch_name', 'pitcher', 'batter', 'release_speed', 'release_spin_rate', 'zone_sc', 'type', 'description', 'events', 'bb_type']

Muestra (velocidad, xwOBA, barrels):
   inning  at_bat_index  pitch_number  start_speed  launch_speed  estimated_woba_using_speedangle  launch_speed_angle
0       1             0             1         92.5           NaN                              NaN                 NaN
1       1             0             2         90.5           NaN                              NaN                 NaN
2       1             0             3         82.9           NaN                              NaN                 NaN
3       1             0             4  

In [75]:
# Paso 2: Pipeline propuesto y patrón "content extractor" (como LIDOM)
# LIDOM: load_game_data(game_id, game_date) → dict con linescore, batting, pitching
# MLB:  load_game_pitches(game_pk) → DataFrame enriched listo para contenido

def load_game_pitches(game_pk: int, warehouse_base: Path = None) -> pd.DataFrame:
    """
    Carga pitches enriquecidos (feed + Statcast) para un juego.
    
    Uso típico para contenido:
        df = load_game_pitches(776498)
        barrels = df[df["launch_speed_angle"] == 2]
        max_velo = df["start_speed"].max()
    """
    from pybaseball import statcast_single_game
    
    base = warehouse_base or Path("../outputs")  # futuro: data/warehouse/mlb/{year}/{stage}/raw
    # Nombres posibles: sample_feed_live_{pk}.json (pruebas) o game_{pk}_{date}_feed_live.json (warehouse)
    raw_path = base / f"sample_feed_live_{game_pk}.json"
    
    if not raw_path.exists():
        raise FileNotFoundError(f"Raw feed no encontrado: {raw_path}")
    
    with open(raw_path) as f:
        feed = json.load(f)
    
    all_plays = feed["liveData"]["plays"]["allPlays"]
    pitches_list = []
    for play in all_plays:
        ab_idx = play["atBatIndex"]
        inn = play["about"]["inning"]
        half = play["about"]["halfInning"]
        for ev in play.get("playEvents", []):
            if not ev.get("isPitch") or "pitchData" not in ev:
                continue
            pd_ = ev["pitchData"]
            row = {
                "game_pk": game_pk, "inning": inn, "half_inning": half,
                "at_bat_index": ab_idx, "pitch_number": ev.get("pitchNumber"),
                "start_speed": pd_.get("startSpeed"), "zone": pd_.get("zone"),
                "play_id": ev.get("playId"),
            }
            if "breaks" in pd_:
                row["spin_rate"] = pd_["breaks"].get("spinRate")
            if "hitData" in ev:
                hd = ev["hitData"]
                row["launch_speed"] = hd.get("launchSpeed")
                row["launch_angle"] = hd.get("launchAngle")
            pitches_list.append(row)
    
    df_feed = pd.DataFrame(pitches_list)
    df_sc = statcast_single_game(game_pk)
    
    if df_sc is None or len(df_sc) == 0:
        return df_feed  # Sin Statcast, devolver solo feed
    
    df_sc = df_sc.rename(columns={"at_bat_number": "at_bat_index"})
    _cols = ["game_pk","inning","at_bat_index","pitch_number","estimated_woba_using_speedangle",
             "woba_value","launch_speed_angle","pitch_name","pitcher","batter","release_speed","zone","events"]
    statcast_cols = [c for c in _cols if c in df_sc.columns]
    df_sc_sub = df_sc[statcast_cols]
    
    return df_feed.merge(
        df_sc_sub,
        on=["game_pk", "inning", "at_bat_index", "pitch_number"],
        how="left",
    )

# Probar (usa sample_feed_live_776498.json en outputs)
# df_content = load_game_pitches(776498, warehouse_base=Path("../outputs"))
# df_content.shape

### Resumen: Flujo para contenido (como LIDOM)

| Paso | Acción |
|------|--------|
| 1 | **Ingest:** `python -m src.ingestion.load_mlb_warehouse` — guarda raw intacto + pitches_enriched |
| 2 | **Raw:** `{warehouse}/{year}/{stage}/raw/game_{pk}_{date}_feed_live.json` — **nunca se modifica** |
| 3 | **Pitches enriched:** `{warehouse}/{year}/{stage}/pitches_enriched/game_{pk}_{date}_pitches_enriched.parquet` — feed + Statcast |
| 4 | **Content:** `pd.read_parquet(...)` o `load_game_pitches()` → DataFrame listo para gráficos, tweets |

**--from-raw:** Re-procesa solo pitches_enriched desde raw existente (sin llamar API feed, solo Statcast).

In [76]:
# Prueba rápida del content extractor (ejecutar tras la celda anterior)
try:
    df_content = load_game_pitches(776498, warehouse_base=Path("../outputs"))
    print(f"✅ load_game_pitches(776498): {len(df_content)} pitches, {len(df_content.columns)} columnas")
    if "estimated_woba_using_speedangle" in df_content.columns:
        print("   xwOBA presente, sample:", df_content["estimated_woba_using_speedangle"].dropna().head(3).tolist())
except FileNotFoundError as e:
    print("⚠️", e)

✅ load_game_pitches(776498): 255 pitches, 20 columnas
   xwOBA presente, sample: [0.237, 0.161, 0.607]


### Join feed/live ↔ Statcast (conceptual)

| feed/live | Statcast |
|-----------|----------|
| `game_pk` | `game_pk` |
| `inning` | `inning` |
| `at_bat_index` | `at_bat_number` |
| `pitch_number` | `pitch_number` |

**Qué ganamos con el join:** `estimated_woba_using_speedangle` (xwOBA), `woba_value`, `launch_speed_angle` (barrels), etc.

**Próximos pasos suaves:**
1. Convertir `pitches` a DataFrame y guardar como Parquet (tabla `pitches` en warehouse)
2. Cuando pybaseball funcione: bajar Statcast por juego, hacer el join, enriquecer
3. Bat speed: explorar después (solo Baseball Savant CSV)

## EDA: pitches_enriched

Exploración del archivo `game_776498_20250901_pitches_enriched.parquet` — columnas, tipos, estadísticas básicas y sample de datos.

In [77]:
import pandas as pd
from pathlib import Path

# Cargar pitches_enriched
path_parquet = Path("../data/warehouse/mlb/2025/regular_season/pitches_enriched/game_776498_20250901_pitches_enriched.parquet")
df = pd.read_parquet(path_parquet)

print("=== Shape ===")
print(df.shape)
print(f"\n=== Columnas ({len(df.columns)}) ===")
print(df.columns.tolist())
print("\n=== Tipos (dtypes) ===")
print(df.dtypes)

=== Shape ===
(255, 26)

=== Columnas (26) ===
['game_pk', 'inning', 'half_inning', 'at_bat_index', 'pitch_number', 'start_speed', 'zone', 'play_id', 'spin_rate', 'launch_speed', 'launch_angle', 'estimated_woba_using_speedangle', 'woba_value', 'woba_denom', 'estimated_ba_using_speedangle', 'launch_speed_angle', 'pitch_name', 'pitcher', 'batter', 'release_speed', 'release_spin_rate', 'zone_sc', 'type', 'description', 'events', 'bb_type']

=== Tipos (dtypes) ===
game_pk                              int64
inning                               int64
half_inning                            str
at_bat_index                         int64
pitch_number                         int64
start_speed                        float64
zone                                 int64
play_id                                str
spin_rate                            int64
launch_speed                       float64
launch_angle                       float64
estimated_woba_using_speedangle    float64
woba_value         

In [78]:
# Estadísticas descriptivas numéricas
print("=== describe() ===")
df.describe()

=== describe() ===


,game_pk,inning,at_bat_index,pitch_number,start_speed,zone,spin_rate,launch_speed,launch_angle,estimated_woba_using_speedangle,woba_value,woba_denom,estimated_ba_using_speedangle,launch_speed_angle,pitcher,batter,release_speed,release_spin_rate,zone_sc
count,255.0,255.000000,255.000000,255.000000,255.000000,255.000000,255.000000,40.000000,40.000000,31.000000,31.00000,31.0,27.000000,27.000000,166.000000,166.000000,166.000000,166.000000,166.000000
mean,776498.0,4.717647,29.662745,3.054902,89.282745,8.886275,2476.698039,85.762500,14.475000,0.235205,0.13871,1.0,0.252129,3.000000,681153.939759,695462.927711,89.500000,2472.686747,8.337349
std,0.0,2.500134,17.725931,1.791528,6.032277,4.217372,226.947493,14.158692,30.781269,0.266073,0.38269,0.0,0.251347,1.270978,32700.234889,42099.357109,5.742199,238.619485,4.200367
min,776498.0,1.000000,0.000000,1.000000,76.400000,1.000000,1305.000000,51.800000,-76.000000,0.000000,0.00000,1.0,0.001000,1.000000,656848.000000,605137.000000,76.700000,1305.000000,1.000000
25%,776498.0,2.000000,14.000000,2.000000,84.100000,5.000000,2323.500000,77.950000,-6.000000,0.031500,0.00000,1.0,0.050500,2.000000,669199.000000,672640.000000,84.350000,2330.250000,5.000000
50%,776498.0,5.000000,30.000000,3.000000,89.700000,9.000000,2521.000000,85.500000,16.500000,0.110000,0.00000,1.0,0.182000,3.000000,674841.000000,682663.000000,90.350000,2521.500000,8.000000
75%,776498.0,7.000000,45.000000,4.000000,94.350000,13.000000,2670.500000,97.275000,30.500000,0.316500,0.00000,1.0,0.330500,4.000000,674841.000000,691594.000000,94.400000,2670.500000,13.000000
max,776498.0,9.000000,60.000000,9.000000,100.300000,14.000000,3014.000000,110.600000,70.000000,0.836000,1.60000,1.0,0.889000,6.000000,806188.000000,805300.000000,99.400000,2956.000000,14.000000


In [79]:
# Sample de los datos (primeras filas)
print("=== head(15) ===")
df.head(15)

=== head(15) ===


,game_pk,inning,half_inning,at_bat_index,pitch_number,start_speed,zone,play_id,spin_rate,launch_speed,...,pitch_name,pitcher,batter,release_speed,release_spin_rate,zone_sc,type,description,events,bb_type
0,776498,1,top,0,1,92.5,3,4b10184d-1d8f-33bd-bdc7-c9...,2316,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,776498,1,top,0,2,90.5,11,6ea57801-1e93-3194-b425-e7...,2293,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,776498,1,top,0,3,82.9,2,86eece98-e27c-3d77-b441-e2...,2603,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,776498,1,top,0,4,84.1,4,b0948e04-9bd4-39dd-93b2-4a...,2676,55.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,776498,1,top,1,1,86.3,8,8be1cde2-84e6-3053-9272-9d...,2053,NaN,...,4-Seam Fastball,674841.0,669364.0,92.5,2317.0,3.0,S,called_strike,NaN,NaN
5,776498,1,top,1,2,91.4,12,0deb1050-bb79-3913-8ddd-52...,2362,NaN,...,4-Seam Fastball,674841.0,669364.0,90.5,2293.0,11.0,B,ball,NaN,NaN
6,776498,1,top,1,3,83.3,9,4e1658c7-fb8c-33fd-ac0c-21...,2696,NaN,...,Slider,674841.0,669364.0,82.9,2604.0,2.0,S,called_strike,NaN,NaN
7,776498,1,top,1,4,83.7,13,d0c401c8-ac0e-3030-8b4b-df...,2698,51.8,...,Curveball,674841.0,669364.0,84.1,2677.0,4.0,X,hit_into_play,field_out,ground_ball
8,776498,1,top,2,1,82.2,7,fbe85e6d-5720-33bd-88c6-13...,2690,NaN,...,Changeup,674841.0,682663.0,86.3,2054.0,8.0,B,ball,NaN,NaN
9,776498,1,top,2,2,82.9,7,8059b92e-e60c-3de0-a5d9-f8...,2668,NaN,...,4-Seam Fastball,674841.0,682663.0,91.4,2362.0,12.0,B,ball,NaN,NaN


In [80]:
# Columnas categóricas: conteos y valores únicos
categorical = ["type", "pitch_name", "half_inning", "description", "events", "bb_type"]
for col in categorical:
    if col in df.columns:
        print(f"--- {col} ---")
        print(df[col].value_counts(dropna=False).head(12))
        print()

--- type ---
type
NaN    89
S      74
B      65
X      27
Name: count, dtype: int64

--- pitch_name ---
pitch_name
NaN                89
4-Seam Fastball    64
Slider             42
Sinker             21
Curveball          13
Sweeper            12
Changeup           10
Cutter              3
Split-Finger        1
Name: count, dtype: int64

--- half_inning ---
half_inning
top       151
bottom    104
Name: count, dtype: int64

--- description ---
description
NaN                89
ball               62
called_strike      33
hit_into_play      27
foul               27
swinging_strike    10
blocked_ball        3
foul_tip            3
foul_bunt           1
Name: count, dtype: int64

--- events ---
events
NaN            224
field_out       20
strikeout        4
force_out        2
single           2
triple           1
sac_fly          1
field_error      1
Name: count, dtype: int64

--- bb_type ---
bb_type
NaN            228
ground_ball     13
fly_ball         7
line_drive       4
popup          

### Columnas Statcast (Savant) vía pybaseball

Todas las columnas que devuelve `statcast_single_game()` — organizadas para lectura cómoda.

In [81]:
# Columnas que trae Savant/Statcast (pybaseball) — vista organizada por grupo
from pybaseball import statcast_single_game
import pandas as pd

df_sc = statcast_single_game(776498)
if df_sc is None or len(df_sc) == 0:
    print("⚠️ No hay datos Statcast para ese juego (o requiere red)")
else:
    def _grupo(col):
        if col.startswith("game_"): return "1_game"
        if col.startswith("release_") or col.startswith("plate_"): return "2_pitch_release"
        if col in ("pitcher", "batter", "pitcher_name", "batter_name") or "player" in col: return "3_players"
        if col.startswith("hit_") or col.startswith("launch_") or col.startswith("hc_"): return "4_batted_ball"
        if "woba" in col or "xba" in col or "estimated" in col or col == "launch_speed_angle": return "5_expected"
        if "zone" in col or "type" in col or "description" in col or "stand" in col or "p_throws" in col: return "6_pitch_info"
        if col in ("inning", "at_bat_number", "pitch_number", "inning_topbot", "balls", "strikes"): return "7_count"
        if "break" in col or "spin" in col or "extension" in col: return "8_movement"
        if "sv_id" in col or "play_" in col or "index" in col or col in ("events", "bb_type", "des"): return "9_play"
        return "0_other"

    rows = []
    for c in df_sc.columns:
        s = df_sc[c]
        sample = s.dropna().iloc[0] if s.notna().any() else "—"
        if isinstance(sample, float) and len(str(sample)) > 12:
            sample = f"{sample:.4g}"
        elif isinstance(sample, str) and len(str(sample)) > 28:
            sample = str(sample)[:25] + "..."
        rows.append({"columna": c, "tipo": str(s.dtype), "no_nulos": int(s.notna().sum()), "ejemplo": sample, "_g": _grupo(c)})
    meta = pd.DataFrame(rows).sort_values("_g").drop(columns="_g").reset_index(drop=True)
    meta.insert(0, "#", meta.index + 1)

    print(f"Statcast: {len(df_sc)} pitches, {len(df_sc.columns)} columnas (ordenadas por tema)\n")
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_colwidth", 30)
    display(meta)

Statcast: 255 pitches, 118 columnas (ordenadas por tema)



,#,columna,tipo,no_nulos,ejemplo
0,1,home_team,str,255,WSH
1,2,away_team,str,255,MIA
2,3,pfx_x,float64,255,1.6
3,4,pfx_z,float64,255,0.35
4,5,on_3b,float64,13,676572.0
5,6,on_2b,float64,18,695734.0
6,7,on_1b,float64,49,672640.0
7,8,outs_when_up,int64,255,2
8,9,tfs_deprecated,float64,0,—
9,10,tfs_zulu_deprecated,float64,0,—


## Feed ↔ Statcast Join: Deep Dive

Validamos el join en acción: no hay ID común entre feed y Statcast, pero sí un **composite key**:

```
game_pk + inning + at_bat_index + pitch_number
```

**Ajuste importante:** Statcast usa `at_bat_number` (1-based), el feed usa `atBatIndex` (0-based). Para alinear: `at_bat_index = at_bat_number - 1`.

In [10]:
# Setup
import json
import pandas as pd
from pathlib import Path
from pybaseball import statcast_single_game

GAME_PK = 776498
WAREHOUSE = Path("../data/warehouse/mlb")

In [11]:
# 1. Cargar feed y extraer pitches (con composite key + play_id)
raw_path = next(WAREHOUSE.rglob(f"game_{GAME_PK}_*_feed_live.json"), None)
if raw_path is None:
    raise FileNotFoundError(f"No raw feed para game_pk={GAME_PK}")

with open(raw_path) as f:
    feed = json.load(f)

pitches_feed = []
for play in feed["liveData"]["plays"]["allPlays"]:
    inn = play["about"]["inning"]
    half = play["about"]["halfInning"]
    ab_idx = play["atBatIndex"]
    for ev in play.get("playEvents", []):
        if not ev.get("isPitch") or "pitchData" not in ev:
            continue
        pitches_feed.append({
            "game_pk": GAME_PK,
            "inning": inn,
            "at_bat_index": ab_idx,
            "pitch_number": ev.get("pitchNumber"),
            "play_id": ev.get("playId"),
            "start_speed": ev["pitchData"].get("startSpeed"),
        })
df_feed = pd.DataFrame(pitches_feed)

print(f"Feed: {len(df_feed)} pitches")
print(f"Columnas: {list(df_feed.columns)}")
print(f"\nComposite key sample (primeros 3):")
print(df_feed[["game_pk", "inning", "at_bat_index", "pitch_number", "play_id"]].head(3).to_string())

Feed: 255 pitches
Columnas: ['game_pk', 'inning', 'at_bat_index', 'pitch_number', 'play_id', 'start_speed']

Composite key sample (primeros 3):
   game_pk  inning  at_bat_index  pitch_number                               play_id
0   776498       1             0             1  4b10184d-1d8f-33bd-bdc7-c9ef19a17458
1   776498       1             0             2  6ea57801-1e93-3194-b425-e7c4b8276760
2   776498       1             0             3  86eece98-e27c-3d77-b441-e2581baaf25c


In [12]:
# 2. Cargar Statcast (pybaseball)
df_statcast = statcast_single_game(GAME_PK)
if df_statcast is None or len(df_statcast) == 0:
    raise ValueError(f"Statcast vacío para game_pk={GAME_PK}")

# Statcast usa at_bat_number (1-based); feed usa atBatIndex (0-based)
# Ajuste: at_bat_index = at_bat_number - 1 para alinear con el feed
df_statcast = df_statcast.copy()
df_statcast["at_bat_index"] = df_statcast["at_bat_number"] - 1

key_cols = ["game_pk", "inning", "at_bat_index", "pitch_number"]
print(f"Statcast: {len(df_statcast)} pitches")
print(f"\nComposite key sample (primeros 3):")
print(df_statcast[key_cols].head(3).to_string())

Statcast: 255 pitches

Composite key sample (primeros 3):
    game_pk  inning  at_bat_index  pitch_number
80   776498       9            60             1
82   776498       9            59             6
89   776498       9            59             5


In [13]:
# 3. Verificar alineación del composite key (antes del merge)
feed_keys = df_feed.set_index(key_cols).index
sc_keys = df_statcast.set_index(key_cols).index

in_both = feed_keys.intersection(sc_keys)
only_feed = feed_keys.difference(sc_keys)
only_statcast = sc_keys.difference(feed_keys)

print("=== Validación del composite key ===")
print(f"Pitches en feed:     {len(df_feed)}")
print(f"Pitches en Statcast: {len(df_statcast)}")
print(f"Matchean (en ambos): {len(in_both)}")
print(f"Solo en feed:        {len(only_feed)}")
print(f"Solo en Statcast:    {len(only_statcast)}")
print(f"\nMatch rate: {100 * len(in_both) / len(df_feed):.1f}% (feed) | {100 * len(in_both) / len(df_statcast):.1f}% (Statcast)")
if len(only_feed) > 0:
    print(f"\n⚠️ Pitches solo en feed (primeros 3):\n{only_feed[:3].tolist()}")
if len(only_statcast) > 0:
    print(f"\n⚠️ Pitches solo en Statcast (primeros 3):\n{only_statcast[:3].tolist()}")

=== Validación del composite key ===
Pitches en feed:     255
Pitches en Statcast: 255
Matchean (en ambos): 255
Solo en feed:        0
Solo en Statcast:    0

Match rate: 100.0% (feed) | 100.0% (Statcast)


In [14]:
# 4. Merge: Statcast base + play_id desde feed
# Solo necesitamos play_id del feed; el resto viene de Statcast
df_feed_ids = df_feed[key_cols + ["play_id"]]
merged = df_statcast.merge(
    df_feed_ids,
    on=key_cols,
    how="left",
)

print(f"Merged: {len(merged)} filas, {len(merged.columns)} columnas")
print(f"play_id no nulos: {merged['play_id'].notna().sum()} (de {len(merged)})")

Merged: 255 filas, 120 columnas
play_id no nulos: 255 (de 255)


In [15]:
# 5. Comparar valores overlap: feed.start_speed vs Statcast.release_speed
tmp = df_feed.merge(df_statcast[key_cols + ["release_speed"]], on=key_cols, how="inner")
tmp["velo_diff"] = abs(tmp["start_speed"] - tmp["release_speed"])
print("=== feed.start_speed vs Statcast.release_speed (inner join) ===")
print(tmp["velo_diff"].describe().to_string())
print(f"\nMax diff: {tmp['velo_diff'].max():.3f} mph")

=== feed.start_speed vs Statcast.release_speed (inner join) ===
count    255.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0

Max diff: 0.000 mph


In [16]:
# 6. Vista final: merged con columnas clave (incl. play_id)
cols_show = key_cols + ["play_id", "release_speed", "pitch_name", "description", "launch_speed", "estimated_woba_using_speedangle"]
cols_show = [c for c in cols_show if c in merged.columns]
merged[cols_show].head(12)

,game_pk,inning,at_bat_index,pitch_number,play_id,release_speed,pitch_name,description,launch_speed,estimated_woba_using_speedangle
0,776498,9,60,1,b3e06834-770e-3596-a35e-59b33e0354fc,90.5,Changeup,hit_into_play,86.2,0.125000
1,776498,9,59,6,8af48ec1-71a0-34d9-a553-3fc486e94ea2,89.2,Slider,swinging_strike,NaN,0.000000
2,776498,9,59,5,7334f814-478b-3d5c-a5bc-265973765e8d,98.8,Sinker,foul,84.5,NaN
3,776498,9,59,4,3ae973a7-e889-30d6-94b5-62bbf5e8b661,100.3,Sinker,foul,NaN,NaN
4,776498,9,59,3,d6eaa9ca-a54a-3afe-922d-f2dfc8497653,98.8,Sinker,ball,NaN,NaN
5,776498,9,59,2,573af184-7cc4-3cca-aa13-e2ff6921ccee,99.4,Sinker,ball,NaN,NaN
6,776498,9,59,1,6fb5b68f-089e-33fb-86f5-c30aaf0ab6a0,98.6,Sinker,foul,94.0,NaN
7,776498,9,58,8,722987b1-8c01-3622-bb1a-8a6dc855695a,100.1,Sinker,ball,NaN,0.691497
8,776498,9,58,7,50bc1af3-6e5d-3f2f-a5e7-af3d2d155cde,88.6,Changeup,ball,NaN,NaN
9,776498,9,58,6,cf05e523-fe82-3018-a6fc-ab33d8591533,91.2,Slider,foul,NaN,NaN


In [19]:
# Verificar que play_id corresponde al pitch correcto (trazar al raw)
# Re-cargamos feed desde raw para evitar scope/stale
_raw_path = next(WAREHOUSE.rglob(f"game_{GAME_PK}_*_feed_live.json"), None)
_feed = json.load(open(_raw_path)) if _raw_path else feed

# Normalizar description: feed ("Ball", "In play, out(s)") -> Statcast ("ball", "hit_into_play")
FEED_DESC_MAP = {
    "in_play,_out(s)": "hit_into_play", "in_play,_run(s)": "hit_into_play", "in_play,_no_out": "hit_into_play",
    "ball": "ball", "called_strike": "called_strike", "swinging_strike": "swinging_strike",
    "foul": "foul", "foul_tip": "foul_tip", "foul_bunt": "foul_bunt",
    "blocked_ball": "blocked_ball", "ball_in_dirt": "blocked_ball",  # feed: ball in dirt
    "swinging_strike_blocked": "swinging_strike_blocked", "swinging_strike_(blocked)": "swinging_strike_blocked",
    "hit_by_pitch": "hit_by_pitch", "missed_bunt": "missed_bunt",
}
def _norm_desc(d):
    if pd.isna(d) or not d: return None
    s = str(d).lower().replace(" ", "_").replace("-", "_")
    return FEED_DESC_MAP.get(s, s)

raw_lookup = {}
for play in _feed["liveData"]["plays"]["allPlays"]:
    inn = play["about"]["inning"]
    ab_idx = play["atBatIndex"]
    bat_id = play.get("matchup", {}).get("batter", {}).get("id")
    pit_id = play.get("matchup", {}).get("pitcher", {}).get("id")
    for ev in play.get("playEvents", []):
        if not ev.get("isPitch") or "pitchData" not in ev:
            continue
        pn = ev.get("pitchNumber")
        k = (int(inn), int(ab_idx), int(pn))
        raw_lookup[k] = {
            "play_id": ev.get("playId"),
            "start_speed": ev["pitchData"].get("startSpeed"),
            "batter_id": bat_id,
            "pitcher_id": pit_id,
            "description": _norm_desc(ev.get("details", {}).get("description")),
        }

def _lookup(row):
    k = (int(row["inning"]), int(row["at_bat_index"]), int(row["pitch_number"]))
    return raw_lookup.get(k, {})

merged["_raw"] = merged.apply(lambda r: _lookup(r), axis=1)
merged["_raw_play_id"] = merged["_raw"].apply(lambda x: x.get("play_id"))
merged["_raw_start_speed"] = merged["_raw"].apply(lambda x: x.get("start_speed"))
merged["_raw_batter"] = merged["_raw"].apply(lambda x: x.get("batter_id"))
merged["_raw_pitcher"] = merged["_raw"].apply(lambda x: x.get("pitcher_id"))
merged["_raw_desc"] = merged["_raw"].apply(lambda x: x.get("description"))

play_id_ok = (merged["play_id"] == merged["_raw_play_id"]).sum()
velo_ok = ((merged["_raw_start_speed"].notna()) & (abs(merged["release_speed"] - merged["_raw_start_speed"]) < 0.01)).sum()
batter_ok = (merged["batter"].fillna(0).astype(int) == merged["_raw_batter"].fillna(0).astype(int)).sum()
pitcher_ok = (merged["pitcher"].fillna(0).astype(int) == merged["_raw_pitcher"].fillna(0).astype(int)).sum()
desc_ok = (merged["description"].fillna("").astype(str).str.strip() == merged["_raw_desc"].fillna("").astype(str).str.strip()).sum()
n_found = merged["_raw_play_id"].notna().sum()

# Diagnosticar description mismatches
desc_mismatch = merged[merged["description"].fillna("").astype(str).str.strip() != merged["_raw_desc"].fillna("").astype(str).str.strip()]
if len(desc_mismatch) > 0:
    print("=== Description mismatches (feed vs Statcast) ===")
    cols_diag = ["inning", "at_bat_index", "pitch_number", "_raw_desc", "description", "pitch_name"]
    cols_diag = [c for c in cols_diag if c in desc_mismatch.columns]
    print(desc_mismatch[cols_diag].to_string())

merged.drop(columns=["_raw", "_raw_play_id", "_raw_start_speed", "_raw_batter", "_raw_pitcher", "_raw_desc"], inplace=True)

print("=== Validación robusta del merge (feed vs merged) ===")
print(f"Keys en raw_lookup: {len(raw_lookup)} | merged rows: {len(merged)}")
print(f"Rows encontradas en raw: {n_found}/{len(merged)}")
print()
print("Comparación campo a campo:")
print(f"  play_id:        {play_id_ok}/{len(merged)} ({100*play_id_ok/len(merged):.1f}%)")
print(f"  release_speed:  {velo_ok}/{len(merged)} ({100*velo_ok/len(merged):.1f}%)")
print(f"  batter_id:      {batter_ok}/{len(merged)} ({100*batter_ok/len(merged):.1f}%)")
print(f"  pitcher_id:     {pitcher_ok}/{len(merged)} ({100*pitcher_ok/len(merged):.1f}%)")
print(f"  description:    {desc_ok}/{len(merged)} ({100*desc_ok/len(merged):.1f}%)")
all_ok = play_id_ok == velo_ok == batter_ok == pitcher_ok == desc_ok == len(merged)
if all_ok:
    print("\n✅ Merge 100%% verídico: todos los campos coinciden con el raw")
else:
    print("\n⚠️ Hay discrepancias — revisar campos que no matchean")

=== Validación robusta del merge (feed vs merged) ===
Keys en raw_lookup: 255 | merged rows: 255
Rows encontradas en raw: 255/255

Comparación campo a campo:
  play_id:        255/255 (100.0%)
  release_speed:  255/255 (100.0%)
  batter_id:      255/255 (100.0%)
  pitcher_id:     255/255 (100.0%)
  description:    255/255 (100.0%)

✅ Merge 100%% verídico: todos los campos coinciden con el raw


### Cómo coinciden los play_id — explicación visual

```
RAW FEED (JSON)                          MERGED (Statcast + play_id)
─────────────────────────────────────────────────────────────────────

allPlays[0]  (1er PA, inning 1)
  │
  ├─ playEvents[0]  pitch 1
  │     playId: "4b10184d-1d8f-33bd-bdc7-c9ef19a17458"
  │     startSpeed: 92.5
  │     ───────────────────►  KEY: (inning=1, at_bat_index=0, pitch_number=1)
  │                                                  │
  ├─ playEvents[1]  pitch 2                          │
  │     playId: "6ea57801-1e93-3194-b425-e7c4b8276760"    │
  │     startSpeed: 90.5                             │
  │     ───────────────────►  KEY: (1, 0, 2)         │
  │                                                  │
  └─ ...                                             │
                                                     │
STATCAST (sin play_id)                                │
  Row: inning=1, at_bat_index=0, pitch_number=1       │
       release_speed=92.5, pitch_name="Changeup"      │
       ──────────────────────────────────────────────┘
                         │
                         ▼
              MERGE por KEY → misma fila
                         │
                         ▼
  MERGED: inning=1, at_bat_index=0, pitch_number=1
          release_speed=92.5, pitch_name="Changeup"
          play_id="4b10184d-1d8f-33bd-bdc7-c9ef19a17458"  ← viene del feed
```

El **composite key** (inning, at_bat_index, pitch_number) identifica unívocamente cada pitch. El merge asocia la fila de Statcast con la fila del feed que tiene la misma key, y copia el play_id. Por eso coinciden.

In [9]:
# Demostración: tomar 1 pitch y mostrar que play_id coincide
# Pitch: inning 1, at_bat 0, pitch 1 (el primer pitch del juego)
key_demo = (1, 0, 1)

# Del raw (ya tenemos raw_lookup de la celda anterior)
raw_val = raw_lookup.get(key_demo, {})

# Del merged (buscar la fila con esa key)
row = merged[(merged["inning"] == key_demo[0]) & (merged["at_bat_index"] == key_demo[1]) & (merged["pitch_number"] == key_demo[2])].iloc[0]

print("=" * 60)
print("PITCH: inning 1, at_bat 0, pitch 1 (primer pitch del juego)")
print("=" * 60)
print("\n📁 RAW FEED (extraído del JSON):")
print(f"   play_id:     {raw_val.get('play_id')}")
print(f"   start_speed: {raw_val.get('start_speed')} mph")
print("\n📊 MERGED (Statcast + feed):")
print(f"   play_id:       {row['play_id']}")
print(f"   release_speed: {row['release_speed']} mph")
print(f"   pitch_name:    {row['pitch_name']}")
print("\n✅ ¿Coinciden?")
print(f"   play_id:   {raw_val.get('play_id') == row['play_id']}")
print(f"   velocidad: {abs((raw_val.get('start_speed') or 0) - row['release_speed']) < 0.01}")

PITCH: inning 1, at_bat 0, pitch 1 (primer pitch del juego)

📁 RAW FEED (extraído del JSON):
   play_id:     4b10184d-1d8f-33bd-bdc7-c9ef19a17458
   start_speed: 92.5 mph

📊 MERGED (Statcast + feed):
   play_id:       4b10184d-1d8f-33bd-bdc7-c9ef19a17458
   release_speed: 92.5 mph
   pitch_name:    4-Seam Fastball

✅ ¿Coinciden?
   play_id:   True
   velocidad: True
